# Experiment 5: PCA and t-SNE dimensionality reduction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

digits = load_digits()           # 64 features (8x8 pixels)
X, y = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# Accuracy before reduction
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
clf.fit(X_tr, y_tr)
print("Accuracy with 64 features:", round(clf.score(X_te, y_te), 3))

In [ ]:
# PCA to 2D, and PCA keeping 95% of the variance
pca2 = PCA(n_components=2).fit(X)
Z_pca = pca2.transform(X)
print("Variance explained by 2 components:", round(pca2.explained_variance_ratio_.sum(), 3))

pca95 = PCA(n_components=0.95).fit(X_tr)
clf95 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
clf95.fit(pca95.transform(X_tr), y_tr)
print("Components kept for 95% variance:", pca95.n_components_)
print("Accuracy after PCA:", round(clf95.score(pca95.transform(X_te), y_te), 3))

In [ ]:
# t-SNE on a 1000-sample subset (it is slow on the full dataset)
idx = np.random.default_rng(0).choice(len(X), 1000, replace=False)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto",
              random_state=0).fit_transform(X[idx])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
sc = ax[0].scatter(Z_pca[:, 0], Z_pca[:, 1], c=y, cmap="tab10", s=8)
ax[0].set_title("PCA (2D)")
ax[1].scatter(Z_tsne[:, 0], Z_tsne[:, 1], c=y[idx], cmap="tab10", s=8)
ax[1].set_title("t-SNE (2D)")
fig.colorbar(sc, ax=ax, label="digit")
plt.show()